### Foldername extractor

In [8]:
import os
import csv

# Path to the directory you want to scan
folder_path = '/Users/maxbreitruck/Library/CloudStorage/OneDrive-UniversitaetSt.Gallen/001_IWI/Dissertation/Paper#2/Yukkalab Daten/Analysis_Data_Final_STOXX600'

# Path to the CSV file where you want to save the folder names
csv_file_path = '/Users/maxbreitruck/Downloads/Yukka_Stoxx600.csv'

# List to hold all subfolder names
subfolder_names = []

# Walk through the directory
for root, dirs, files in os.walk(folder_path):
    # Add all directory names to the list
    subfolder_names.extend(dirs)
    # Prevent descending into subdirectories
    break

# Write subfolder names to the CSV file
with open(csv_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Write a header (optional)
    writer.writerow(['Subfolder Name'])
    # Write each subfolder name in a separate row
    for name in subfolder_names:
        writer.writerow([name])

print(f'Successfully written the names of subfolders in {folder_path} to {csv_file_path}')


Successfully written the names of subfolders in /Users/maxbreitruck/Library/CloudStorage/OneDrive-UniversitaetSt.Gallen/001_IWI/Dissertation/Paper#2/Yukkalab Daten/Analysis_Data_Final_STOXX600 to /Users/maxbreitruck/Downloads/Yukka_Stoxx600.csv


In [9]:
import pandas as pd

yukka_daten_df = pd.read_csv('/Users/maxbreitruck/Downloads/Yukka_Stoxx600.csv')
stoxx600_excel_df = pd.read_excel('/Users/maxbreitruck/Downloads/SXXGR.xlsx')

# Extract company names from the 'Subfolder Name' in Yukka Daten CSV

In [11]:
import pandas as pd
import difflib

def rough_match(yukka_name, stoxx_names):
    """Function to perform rough matching based on partial string similarity."""
    for stoxx_name in stoxx_names:
        if yukka_name in stoxx_name or stoxx_name in yukka_name:
            return stoxx_name
    return None

# Load the Yukka Stoxx600 CSV file
yukka_stoxx600_df = pd.read_csv('/Users/maxbreitruck/Downloads/Yukka_Stoxx600.csv')

# Load the STOXX Europe 600 Excel file
stoxx600_excel_df = pd.read_excel('/Users/maxbreitruck/Downloads/SXXGR.xlsx')

# Normalize company names for rough matching
yukka_stoxx600_df['Normalized Company Name'] = yukka_stoxx600_df['Subfolder Name'].apply(lambda x: x.split('_')[0].lower())
stoxx600_excel_df['Normalized Company Name'] = stoxx600_excel_df['Company'].str.lower()

# Perform the rough matching
yukka_stoxx600_df['Rough Matched Company Name'] = yukka_stoxx600_df['Normalized Company Name'].apply(
    lambda name: rough_match(name, stoxx600_excel_df['Normalized Company Name'])
)

# Map the matched company name to its corresponding sector and company name from STOXX 600
stoxx600_mapping_dict = stoxx600_excel_df.set_index('Normalized Company Name')['Supersector'].to_dict()
stoxx600_name_dict = stoxx600_excel_df.set_index('Normalized Company Name')['Company'].to_dict()

# Retrieve the matched sector and company name
yukka_stoxx600_df['Rough Matched Supersector'] = yukka_stoxx600_df['Rough Matched Company Name'].map(stoxx600_mapping_dict)
yukka_stoxx600_df['Rough Matched Company Name (Full)'] = yukka_stoxx600_df['Rough Matched Company Name'].map(stoxx600_name_dict)

# Drop unnecessary columns
yukka_stoxx600_df.drop(columns=['Normalized Company Name', 'Rough Matched Company Name'], inplace=True)

# Save the updated DataFrame to a new CSV file
yukka_stoxx600_df.to_csv('updated_yukka_stoxx600_with_supersectors.csv', index=False)

print("Matching completed and file saved as 'updated_yukka_stoxx600_with_supersectors.csv'")


Matching completed and file saved as 'updated_yukka_stoxx600_with_supersectors.csv'


In [12]:
import pandas as pd
import difflib

def definite_match(yukka_name, stoxx_names):
    """Function for definite matching based on exact string match."""
    return stoxx_names.get(yukka_name)

def rough_match(yukka_name, stoxx_names):
    """Function for rough matching based on partial string similarity."""
    matches = difflib.get_close_matches(yukka_name, stoxx_names.keys(), n=1, cutoff=0.6)
    return matches[0] if matches else None

# Load the Yukka Stoxx600 CSV file
yukka_stoxx600_df = pd.read_csv('/Users/maxbreitruck/Downloads/Yukka_Stoxx600.csv')

# Load the STOXX Europe 600 Excel file
stoxx600_excel_df = pd.read_excel('/Users/maxbreitruck/Downloads/SXXGR.xlsx')

# Normalize company names for matching
yukka_stoxx600_df['Normalized Company Name'] = yukka_stoxx600_df['Subfolder Name'].apply(lambda x: x.split('_')[0].lower())
stoxx600_excel_df['Normalized Company Name'] = stoxx600_excel_df['Company'].str.lower()

# Create dictionaries for definite and rough matching
stoxx600_definite_dict = stoxx600_excel_df.set_index('Normalized Company Name').to_dict('index')
stoxx600_rough_dict = {row['Normalized Company Name']: row for _, row in stoxx600_excel_df.iterrows()}

# Perform the definite and rough matching
yukka_stoxx600_df['Definite Matched Company Name'] = yukka_stoxx600_df['Normalized Company Name'].apply(
    lambda name: definite_match(name, stoxx600_definite_dict)
)
yukka_stoxx600_df['Rough Matched Company Name'] = yukka_stoxx600_df['Normalized Company Name'].apply(
    lambda name: rough_match(name, stoxx600_rough_dict)
)

# Retrieve the matched sector and company name for both matchings
yukka_stoxx600_df['Definite Matched Supersector'] = yukka_stoxx600_df['Definite Matched Company Name'].apply(
    lambda name: stoxx600_definite_dict.get(name, {}).get('Supersector')
)
yukka_stoxx600_df['Rough Matched Supersector'] = yukka_stoxx600_df['Rough Matched Company Name'].apply(
    lambda name: stoxx600_rough_dict.get(name, {}).get('Supersector')
)

# Drop unnecessary columns
yukka_stoxx600_df.drop(columns=['Normalized Company Name', 'Definite Matched Company Name', 'Rough Matched Company Name'], inplace=True)

# Save the updated DataFrame to a new CSV file
yukka_stoxx600_df.to_csv('updated_yukka_stoxx600_with_both_matchings.csv', index=False)

print("Matching completed and file saved as 'updated_yukka_stoxx600_with_both_matchings.csv'")


ValueError: DataFrame index must be unique for orient='index'.